# ⭐ Day 48: Advanced Outlier & Anomaly Detection  
## Techniques & Visualizations for Robust AI & ML  
### 🚀 Day 48 of 369-Day Python & AI Learning Path

## Welcome to Day 48! 🎯

Today we dive deep into one of the most critical yet often underestimated aspects of machine learning: **advanced outlier and anomaly detection**. While basic statistical methods like IQR and Z-score serve as good starting points, real-world AI/ML projects demand more sophisticated approaches.

### Why This Matters 🔍

Outliers and anomalies are not just statistical nuisances—they can represent:
- **Fraudulent transactions** in financial systems 💳
- **Equipment failures** in industrial IoT sensors ⚙️
- **Rare disease cases** in healthcare diagnostics 🏥
- **Cybersecurity breaches** in network traffic 🛡️
- **Data quality issues** that corrupt model training 📊

### What You'll Learn Today 📚

This notebook goes far beyond basic box plots. We'll explore cutting-edge techniques including Isolation Forests, Local Outlier Factors, density-based clustering for anomaly detection, and multivariate approaches. You'll learn not just *how* to detect anomalies, but *when* to remove them, *when* to keep them, and *how* to build robust ML pipelines that handle real-world data imperfections.

By the end of this session, you'll have a comprehensive toolkit for anomaly detection that rivals production-grade systems used at top tech companies. Let's build detection systems that are as robust as they are intelligent!

## 📋 Table of Contents

1. [Difference Between Outliers and Anomalies](#section1)
2. [Recap of Basic Methods (IQR, Z-Score)](#section2)
3. [Advanced Statistical Methods](#section3)
   - Modified Z-Score
   - Isolation Forest
   - Local Outlier Factor (LOF)
4. [Density-Based Methods (DBSCAN)](#section4)
5. [Machine Learning-Based Anomaly Detection](#section5)
6. [Visualizing Outliers and Anomalies](#section6)
7. [Multivariate Outlier Detection](#section7)
8. [Evaluating Anomaly Detection Performance](#section8)
9. [Handling Strategies](#section9)
10. [Best Practices and Decision Framework](#section10)
11. [🛠️ Hands-On Exercises](#exercises)
12. [Solutions & Key Insights](#solutions)

<a id='section1'></a>
## 1. Difference Between Outliers and Anomalies 🔍

While often used interchangeably, these terms have subtle but important distinctions:

| Aspect | Outliers | Anomalies |
|--------|----------|-----------|
| **Definition** | Statistical deviation from the norm | Pattern that doesn't conform to expected behavior |
| **Context** | Often univariate | Usually multivariate/pattern-based |
| **Interpretation** | Can be noise OR signal | Typically indicates something meaningful |
| **Examples** | Extreme house prices, tall individuals | Fraudulent transactions, machine failures |
| **Handling** | May remove or transform | Usually investigate and model separately |

### 💡 Key Insight
All anomalies are outliers, but not all outliers are anomalies. An outlier might just be a legitimate extreme value, while an anomaly represents a fundamentally different data-generating process.

In [ ]:
# =============================================================================
# Setup and Imports
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set style for professional visualizations
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
np.random.seed(42)

print("✅ All libraries imported successfully!")
print("📦 Ready to build robust anomaly detection systems!")

In [ ]:
# =============================================================================
# Generate Realistic Dataset with Injected Anomalies
# =============================================================================
# Creating a credit card transaction-like dataset with fraud injection

def generate_transaction_data(n_normal=1000, n_fraud=50):
    """
    Generate synthetic credit card transaction data with fraud anomalies.
    """
    # Normal transactions: legitimate spending patterns
    normal_data = {
        'transaction_amount': np.random.lognormal(4, 0.5, n_normal),  # Skewed distribution
        'transaction_time': np.random.uniform(0, 24, n_normal),  # Hours in day
        'merchant_risk_score': np.random.beta(2, 5, n_normal),  # Low risk merchants
        'customer_age': np.random.normal(35, 10, n_normal).clip(18, 80),
        'account_age_days': np.random.exponential(365, n_normal),
        'transactions_per_day': np.random.poisson(3, n_normal),
        'is_fraud': 0
    }
    
    # Fraudulent transactions: anomalous patterns
    fraud_data = {
        'transaction_amount': np.random.lognormal(6, 1.2, n_fraud),  # Higher amounts
        'transaction_time': np.random.choice([0, 1, 2, 3, 4, 5], n_fraud),  # Odd hours
        'merchant_risk_score': np.random.beta(5, 2, n_fraud),  # High risk merchants
        'customer_age': np.random.normal(35, 10, n_fraud).clip(18, 80),
        'account_age_days': np.random.exponential(30, n_fraud),  # New accounts
        'transactions_per_day': np.random.poisson(15, n_fraud),  # Unusual frequency
        'is_fraud': 1
    }
    
    df_normal = pd.DataFrame(normal_data)
    df_fraud = pd.DataFrame(fraud_data)
    
    df = pd.concat([df_normal, df_fraud], ignore_index=True)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # Shuffle
    
    return df

# Generate the dataset
df = generate_transaction_data(n_normal=1000, n_fraud=50)

print(f"📊 Dataset Shape: {df.shape}")
print(f"🔍 Fraud Rate: {df['is_fraud'].mean():.2%}")
print("\n📋 First 5 rows:")
print(df.head())
print("\n📈 Data Description:")
print(df.describe())

<a id='section2'></a>
## 2. Recap of Basic Methods (IQR, Z-Score) 📉

Before diving into advanced techniques, let's quickly review the foundational methods that every data scientist should know.

In [ ]:
# =============================================================================
# Basic Outlier Detection: Z-Score and IQR Methods
# =============================================================================

def detect_outliers_zscore(data, column, threshold=3):
    """Detect outliers using Z-Score method."""
    z_scores = np.abs(stats.zscore(data[column]))
    return z_scores > threshold

def detect_outliers_iqr(data, column):
    """Detect outliers using IQR method."""
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return (data[column] < lower_bound) | (data[column] > upper_bound)

# Apply basic methods on transaction_amount
feature = 'transaction_amount'

df['zscore_outlier'] = detect_outliers_zscore(df, feature, threshold=3)
df['iqr_outlier'] = detect_outliers_iqr(df, feature)

print(f"📊 Z-Score Outliers (|z| > 3): {df['zscore_outlier'].sum()} ({df['zscore_outlier'].mean():.1%})")
print(f"📊 IQR Outliers: {df['iqr_outlier'].sum()} ({df['iqr_outlier'].mean():.1%})")
print(f"🎯 Actual Fraud Cases: {df['is_fraud'].sum()}")

# Visualize basic methods
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Box plot
sns.boxplot(y=df[feature], ax=axes[0], color='skyblue')
axes[0].set_title('Box Plot: IQR Method Visualization', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Transaction Amount ($)')

# Histogram with Z-score thresholds
axes[1].hist(df[feature], bins=50, alpha=0.7, color='lightcoral', edgecolor='black')
mean_val = df[feature].mean()
std_val = df[feature].std()
axes[1].axvline(mean_val + 3*std_val, color='red', linestyle='--', linewidth=2, label='Z=+3')
axes[1].axvline(mean_val - 3*std_val, color='red', linestyle='--', linewidth=2, label='Z=-3')
axes[1].set_title('Distribution: Z-Score Method', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Transaction Amount ($)')
axes[1].set_ylabel('Frequency')
axes[1].legend()

# Comparison scatter
axes[2].scatter(df.index, df[feature], c=df['is_fraud'], cmap='coolwarm', alpha=0.6, s=30)
axes[2].set_title('Actual Fraud Labels (Ground Truth)', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Sample Index')
axes[2].set_ylabel('Transaction Amount ($)')

plt.tight_layout()
plt.show()

print("\n💡 Observation: Basic methods catch extreme values but may miss sophisticated fraud patterns!")

### ⚠️ Limitations of Basic Methods

1. **Z-Score**: Assumes normal distribution; fails with skewed data (like our transaction amounts)
2. **IQR**: Only considers univariate relationships; misses multivariate fraud patterns
3. **Both**: Treat outliers as noise to remove, rather than anomalies to investigate

> **Recommendation**: Use these as quick sanity checks, but rely on advanced methods for production systems.

<a id='section3'></a>
## 3. Advanced Statistical Methods 🛡️

### 3.1 Modified Z-Score (MAD-based)
More robust than standard Z-score, using Median Absolute Deviation instead of standard deviation.

In [ ]:
# =============================================================================
# Modified Z-Score using Median Absolute Deviation (MAD)
# =============================================================================

def modified_z_score(data):
    """
    Calculate modified Z-score using Median Absolute Deviation.
    More robust to outliers than standard Z-score.
    """
    median = np.median(data)
    mad = np.median(np.abs(data - median))
    # Constant 0.6745 makes MAD comparable to standard deviation for normal data
    modified_z = 0.6745 * (data - median) / mad
    return np.abs(modified_z)

# Apply modified Z-score
df['modified_zscore'] = modified_z_score(df['transaction_amount'])
df['modified_z_outlier'] = df['modified_zscore'] > 3.5

print("🔍 Modified Z-Score Results:")
print(f"   Outliers detected: {df['modified_z_outlier'].sum()}")
print(f"   Fraud cases caught: {df[df['modified_z_outlier']]['is_fraud'].sum()}")
print(f"   Precision: {df[df['modified_z_outlier']]['is_fraud'].mean():.2%}")

# Visualize Modified Z-Score
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Modified Z-score distribution
axes[0].hist(df['modified_zscore'], bins=50, alpha=0.7, color='mediumseagreen', edgecolor='black')
axes[0].axvline(3.5, color='red', linestyle='--', linewidth=2, label='Threshold = 3.5')
axes[0].set_title('Modified Z-Score Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Modified Z-Score')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Comparison: Standard vs Modified Z-Score
comparison_df = pd.DataFrame({
    'Standard Z-Score': df['zscore_outlier'],
    'Modified Z-Score': df['modified_z_outlier'],
    'Actual Fraud': df['is_fraud']
})
comparison_counts = comparison_df.sum()
x_pos = np.arange(len(comparison_counts))
bars = axes[1].bar(x_pos, comparison_counts.values, color=['skyblue', 'mediumseagreen', 'coral'])
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(comparison_counts.index, rotation=15)
axes[1].set_title('Outlier Detection Comparison', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Count')
for bar in bars:
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

### 3.2 Isolation Forest 🌲

Isolation Forest is an ensemble method that isolates anomalies instead of profiling normal data points. It works on the principle that anomalies are "few and different"—easier to isolate than normal points.

In [ ]:
# =============================================================================
# Isolation Forest for Anomaly Detection
# =============================================================================

# Prepare features (exclude target)
feature_cols = ['transaction_amount', 'transaction_time', 'merchant_risk_score', 
                'customer_age', 'account_age_days', 'transactions_per_day']
X = df[feature_cols].copy()

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit Isolation Forest
# contamination parameter approximates the proportion of outliers
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.05,  # Expect ~5% outliers
    random_state=42,
    n_jobs=-1
)

df['iso_forest_outlier'] = iso_forest.fit_predict(X_scaled)
df['iso_forest_outlier'] = df['iso_forest_outlier'] == -1  # -1 indicates outlier
df['iso_forest_score'] = iso_forest.decision_function(X_scaled)  # Anomaly score

print("🌲 Isolation Forest Results:")
print(f"   Anomalies detected: {df['iso_forest_outlier'].sum()}")
print(f"   Fraud cases caught: {df[df['iso_forest_outlier']]['is_fraud'].sum()}")
print(f"   Precision: {df[df['iso_forest_outlier']]['is_fraud'].mean():.2%}")


# Visualize Isolation Forest results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Anomaly scores distribution
axes[0].hist(df['iso_forest_score'], bins=50, alpha=0.7, color='mediumpurple', edgecolor='black')
axes[0].axvline(0, color='red', linestyle='--', linewidth=2, label='Decision Boundary')
axes[0].set_title('Isolation Forest Anomaly Scores', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Anomaly Score (lower = more anomalous)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# 2D scatter plot highlighting anomalies
scatter = axes[1].scatter(df['transaction_amount'], df['transactions_per_day'], 
                         c=df['iso_forest_outlier'], cmap='coolwarm', alpha=0.6, s=40)
axes[1].set_title('Isolation Forest: Detected Anomalies', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Transaction Amount ($)')
axes[1].set_ylabel('Transactions per Day')
plt.colorbar(scatter, ax=axes[1], label='Is Anomaly')

plt.tight_layout()
plt.show()

### 3.3 Local Outlier Factor (LOF) 📍

LOF detects anomalies based on local density deviation. It identifies points that have substantially lower density than their neighbors—perfect for detecting fraud that might blend in globally but stands out locally.

In [ ]:
# =============================================================================
# Local Outlier Factor (LOF)
# =============================================================================

# Fit LOF
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.05,
    novelty=False,
    n_jobs=-1
)

df['lof_outlier'] = lof.fit_predict(X_scaled)
df['lof_outlier'] = df['lof_outlier'] == -1  # -1 indicates outlier
df['lof_score'] = -lof.negative_outlier_factor_  # Convert to positive scores

print("📍 Local Outlier Factor Results:")
print(f"   Anomalies detected: {df['lof_outlier'].sum()}")
print(f"   Fraud cases caught: {df[df['lof_outlier']]['is_fraud'].sum()}")
print(f"   Precision: {df[df['lof_outlier']]['is_fraud'].mean():.2%}")

# Visualize LOF results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# LOF scores distribution
axes[0].hist(df['lof_score'], bins=50, alpha=0.7, color='gold', edgecolor='black')
axes[0].set_title('LOF Outlier Scores', fontsize=12, fontweight='bold')
axes[0].set_xlabel('LOF Score (higher = more anomalous)')
axes[0].set_ylabel('Frequency')

# 2D scatter with LOF anomalies
scatter = axes[1].scatter(df['merchant_risk_score'], df['transaction_time'], 
                         c=df['lof_outlier'], cmap='coolwarm', alpha=0.6, s=40)
axes[1].set_title('LOF: Detected Anomalies', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Merchant Risk Score')
axes[1].set_ylabel('Transaction Time (Hour)')
plt.colorbar(scatter, ax=axes[1], label='Is Anomaly')

plt.tight_layout()
plt.show()

print("\n💡 LOF excels at finding local anomalies that might be missed by global methods like Isolation Forest!")

<a id='section4'></a>
## 4. Density-Based Methods: DBSCAN 🔵

DBSCAN (Density-Based Spatial Clustering of Applications with Noise) can identify outliers as points that don't belong to any cluster. It's particularly effective when anomalies form their own sparse regions.

In [ ]:
# =============================================================================
# DBSCAN for Anomaly Detection
# =============================================================================

# Fit DBSCAN
# eps: maximum distance between samples in a neighborhood
# min_samples: minimum points to form a core point
dbscan = DBSCAN(eps=1.5, min_samples=5, n_jobs=-1)
df['dbscan_cluster'] = dbscan.fit_predict(X_scaled)

# Points with label -1 are considered noise/outliers
df['dbscan_outlier'] = df['dbscan_cluster'] == -1

n_clusters = len(set(df['dbscan_cluster'])) - (1 if -1 in df['dbscan_cluster'] else 0)
n_noise = list(df['dbscan_cluster']).count(-1)

print("🔵 DBSCAN Results:")
print(f"   Estimated clusters: {n_clusters}")
print(f"   Noise points (outliers): {n_noise}")
print(f"   Fraud cases in noise: {df[df['dbscan_outlier']]['is_fraud'].sum()}")
print(f"   Precision: {df[df['dbscan_outlier']]['is_fraud'].mean():.2%}")

# Visualize DBSCAN results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cluster visualization (using first 2 PCA components for visualization)
pca_viz = PCA(n_components=2)
X_pca = pca_viz.fit_transform(X_scaled)

# Plot clusters
unique_labels = set(df['dbscan_cluster'])
colors = plt.cm.Spectral(np.linspace(0, 1, len(unique_labels)))
for label, color in zip(unique_labels, colors):
    if label == -1:
        color = [0, 0, 0, 1]  # Black for noise
    mask = df['dbscan_cluster'] == label
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], c=[color], 
                   label=f'Cluster {label}' if label != -1 else 'Noise', 
                   s=30, alpha=0.6)
axes[0].set_title('DBSCAN Clusters & Noise Points', fontsize=12, fontweight='bold')
axes[0].set_xlabel('PCA Component 1')
axes[0].set_ylabel('PCA Component 2')
axes[0].legend()

# Comparison of all methods
methods = ['Z-Score', 'Modified Z', 'Isolation Forest', 'LOF', 'DBSCAN']
precisions = [
    df.loc[df['zscore_outlier'], 'is_fraud'].mean(),
    df.loc[df['modified_z_outlier'], 'is_fraud'].mean(),
    df.loc[df['iso_forest_outlier'], 'is_fraud'].mean(),
    df.loc[df['lof_outlier'], 'is_fraud'].mean(),
    df.loc[df['dbscan_outlier'], 'is_fraud'].mean()
]
recalls = [
    df.loc[df['is_fraud'] == 1, 'zscore_outlier'].mean(),
    df.loc[df['is_fraud'] == 1, 'modified_z_outlier'].mean(),
    df.loc[df['is_fraud'] == 1, 'iso_forest_outlier'].mean(),
    df.loc[df['is_fraud'] == 1, 'lof_outlier'].mean(),
    df.loc[df['is_fraud'] == 1, 'dbscan_outlier'].mean()
]

x = np.arange(len(methods))
width = 0.35
bars1 = axes[1].bar(x - width/2, precisions, width, label='Precision', color='steelblue')
bars2 = axes[1].bar(x + width/2, recalls, width, label='Recall', color='coral')
axes[1].set_ylabel('Score')
axes[1].set_title('Method Comparison: Precision vs Recall', fontsize=12, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(methods, rotation=45, ha='right')
axes[1].legend()
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

<a id='section5'></a>
## 5. Machine Learning-Based Anomaly Detection 🤖

Advanced ML approaches treat anomaly detection as a supervised or semi-supervised learning problem. Here we demonstrate ensemble approaches and probability-based methods.

In [ ]:
# =============================================================================
# Ensemble Anomaly Detection
# =============================================================================

# Create ensemble prediction (majority voting)
df['ensemble_outlier'] = (
    df['iso_forest_outlier'].astype(int) + 
    df['lof_outlier'].astype(int) + 
    df['dbscan_outlier'].astype(int)
) >= 2  # Flag if 2+ methods agree

print("🤖 Ensemble Method Results (2+ methods agree):")
print(f"   Anomalies detected: {df['ensemble_outlier'].sum()}")
print(f"   Fraud cases caught: {df[df['ensemble_outlier']]['is_fraud'].sum()}")
print(f"   Precision: {df[df['ensemble_outlier']]['is_fraud'].mean():.2%}")
print(f"   Recall: {df.loc[df['is_fraud'] == 1, 'ensemble_outlier'].mean():.2%}")

# Detailed comparison table
comparison_table = pd.DataFrame({
    'Method': ['Z-Score', 'Modified Z-Score', 'Isolation Forest', 'LOF', 'DBSCAN', 'Ensemble (2+)'],
    'Detected': [
        df['zscore_outlier'].sum(),
        df['modified_z_outlier'].sum(),
        df['iso_forest_outlier'].sum(),
        df['lof_outlier'].sum(),
        df['dbscan_outlier'].sum(),
        df['ensemble_outlier'].sum()
    ],
    'True_Positives': [
        df[df['zscore_outlier']]['is_fraud'].sum(),
        df[df['modified_z_outlier']]['is_fraud'].sum(),
        df[df['iso_forest_outlier']]['is_fraud'].sum(),
        df[df['lof_outlier']]['is_fraud'].sum(),
        df[df['dbscan_outlier']]['is_fraud'].sum(),
        df[df['ensemble_outlier']]['is_fraud'].sum()
    ]
})
comparison_table['Precision'] = comparison_table['True_Positives'] / comparison_table['Detected']
comparison_table['Recall'] = comparison_table['True_Positives'] / df['is_fraud'].sum()
comparison_table['F1_Score'] = 2 * (comparison_table['Precision'] * comparison_table['Recall']) / \
                               (comparison_table['Precision'] + comparison_table['Recall'])

print("\n📊 Comprehensive Method Comparison:")
print(comparison_table.round(3).to_string(index=False))

<a id='section6'></a>
## 6. Visualizing Outliers and Anomalies 📊

Effective visualization is crucial for understanding and communicating anomaly detection results.

In [ ]:
# =============================================================================
# Advanced Visualizations for Anomaly Detection
# =============================================================================

# 1. Multi-dimensional scatter matrix
fig = plt.figure(figsize=(16, 12))

# Create a 3x3 subplot grid for pairwise comparisons
features_to_plot = ['transaction_amount', 'merchant_risk_score', 'transaction_time', 'transactions_per_day']
n_features = len(features_to_plot)

for i in range(n_features):
    for j in range(n_features):
        if i != j:
            ax = plt.subplot(n_features, n_features, i * n_features + j + 1)
            scatter = ax.scatter(df[features_to_plot[j]], df[features_to_plot[i]], 
                               c=df['ensemble_outlier'], cmap='coolwarm', alpha=0.6, s=20)
            ax.set_xlabel(features_to_plot[j])
            ax.set_ylabel(features_to_plot[i])
            if i == 0 and j == 1:
                ax.set_title('Ensemble Anomaly Detection\n(Red = Anomaly)', fontsize=10)
        else:
            # Diagonal: histograms
            ax = plt.subplot(n_features, n_features, i * n_features + j + 1)
            ax.hist(df[features_to_plot[i]], bins=30, alpha=0.7, color='skyblue', edgecolor='black')
            ax.set_title(features_to_plot[i][:15], fontsize=9)

plt.suptitle('Pairwise Feature Relationships with Anomaly Highlighting', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Dimensionality Reduction for Visualization
# =============================================================================

# PCA Visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# t-SNE Visualization (computationally intensive, sample if needed)
from sklearn.manifold import TSNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# PCA plot
scatter1 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], 
                          c=df['ensemble_outlier'], cmap='coolwarm', alpha=0.7, s=40)
axes[0].set_title('PCA Projection: Anomaly Detection Results', fontsize=12, fontweight='bold')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.colorbar(scatter1, ax=axes[0], label='Is Anomaly')

# t-SNE plot
scatter2 = axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], 
                          c=df['is_fraud'], cmap='RdYlBu_r', alpha=0.7, s=40)
axes[1].set_title('t-SNE Projection: Ground Truth Labels', fontsize=12, fontweight='bold')
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')
plt.colorbar(scatter2, ax=axes[1], label='Is Fraud')

plt.tight_layout()
plt.show()

print(f"📉 PCA Explained Variance Ratio: {pca.explained_variance_ratio_}")
print(f"📉 Total Variance Captured: {pca.explained_variance_ratio_.sum():.2%}")
print("\n💡 t-SNE preserves local structure better than PCA, making it ideal for visualizing clusters!")

<a id='section7'></a>
## 7. Multivariate Outlier Detection 🔗

Real-world anomalies often only appear anomalous when considering multiple variables simultaneously.

In [ ]:
# =============================================================================
# Mahalanobis Distance for Multivariate Outliers
# =============================================================================

from scipy.spatial.distance import mahalanobis

def calculate_mahalanobis(data):
    """
    Calculate Mahalanobis distance for each point.
    Accounts for correlations between variables.
    """
    mean = np.mean(data, axis=0)
    cov = np.cov(data.T)
    cov_inv = np.linalg.inv(cov)
    
    distances = []
    for i in range(len(data)):
        d = mahalanobis(data[i], mean, cov_inv)
        distances.append(d)
    return np.array(distances)

# Calculate Mahalanobis distances
df['mahalanobis_dist'] = calculate_mahalanobis(X_scaled)
df['mahalanobis_outlier'] = df['mahalanobis_dist'] > np.percentile(df['mahalanobis_dist'], 95)

print("🔗 Mahalanobis Distance Results:")
print(f"   Outliers detected (top 5%): {df['mahalanobis_outlier'].sum()}")
print(f"   Fraud cases caught: {df[df['mahalanobis_outlier']]['is_fraud'].sum()}")
print(f"   Precision: {df[df['mahalanobis_outlier']]['is_fraud'].mean():.2%}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mahalanobis distance distribution
axes[0].hist(df['mahalanobis_dist'], bins=50, alpha=0.7, color='teal', edgecolor='black')
threshold = np.percentile(df['mahalanobis_dist'], 95)
axes[0].axvline(threshold, color='red', linestyle='--', linewidth=2, label=f'95th percentile ({threshold:.2f})')
axes[0].set_title('Mahalanobis Distance Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Mahalanobis Distance')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# 3D scatter plot (using PCA components + original feature)
from mpl_toolkits.mplot3d import Axes3D
ax3d = fig.add_subplot(122, projection='3d')
scatter = ax3d.scatter(X_pca[:, 0], X_pca[:, 1], df['transaction_amount'],
                      c=df['mahalanobis_outlier'], cmap='coolwarm', alpha=0.6, s=30)
ax3d.set_title('3D View: Multivariate Outliers', fontsize=12, fontweight='bold')
ax3d.set_xlabel('PC1')
ax3d.set_ylabel('PC2')
ax3d.set_zlabel('Transaction Amount')

plt.tight_layout()
plt.show()

print("\n💡 Mahalanobis distance accounts for feature correlations—critical for multivariate data!")

<a id='section8'></a>
## 8. Evaluating Anomaly Detection Performance 📈

When ground truth labels are available (like our fraud labels), we can rigorously evaluate detection performance.

In [ ]:
# =============================================================================
# Performance Evaluation with Ground Truth
# =============================================================================

from sklearn.metrics import precision_recall_curve, average_precision_score, roc_curve

# Calculate metrics for methods that provide scores
methods_scores = {
    'Isolation Forest': -df['iso_forest_score'],  # Invert so higher = more anomalous
    'LOF': df['lof_score'],
    'Mahalanobis': df['mahalanobis_dist']
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curves
for name, scores in methods_scores.items():
    fpr, tpr, _ = roc_curve(df['is_fraud'], scores)
    auc = roc_auc_score(df['is_fraud'], scores)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})', linewidth=2)

axes[0].plot([0, 1], [0, 1], 'k--', label='Random Classifier')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves: Anomaly Detection Methods', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Precision-Recall Curves
for name, scores in methods_scores.items():
    precision, recall, _ = precision_recall_curve(df['is_fraud'], scores)
    avg_precision = average_precision_score(df['is_fraud'], scores)
    axes[1].plot(recall, precision, label=f'{name} (AP = {avg_precision:.3f})', linewidth=2)

axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()



<a id='section9'></a>
## 9. Handling Strategies: Remove, Cap, Transform, or Model Separately? 🛠️

Once detected, how should you handle anomalies? The strategy depends on the anomaly type and business context.

In [ ]:
# =============================================================================
# Anomaly Handling Strategies Demonstration
# =============================================================================

# Create a copy for demonstrating different strategies
df_original = df.copy()

# Strategy 1: Removal
df_removed = df[~df['ensemble_outlier']].copy()

# Strategy 2: Capping (Winsorization)
df_capped = df.copy()
for col in ['transaction_amount', 'transactions_per_day']:
    lower = df_capped[col].quantile(0.01)
    upper = df_capped[col].quantile(0.99)
    df_capped[col] = df_capped[col].clip(lower, upper)

# Strategy 3: Log Transformation
df_transformed = df.copy()
df_transformed['transaction_amount_log'] = np.log1p(df_transformed['transaction_amount'])

# Strategy 4: Separate modeling flag
df_flagged = df.copy()
df_flagged['is_high_risk'] = df_flagged['ensemble_outlier'].astype(int)

# Visualize before/after
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Original vs Removed
axes[0, 0].scatter(df_original.index, df_original['transaction_amount'], 
                  c=df_original['ensemble_outlier'], cmap='coolwarm', alpha=0.6, s=20)
axes[0, 0].set_title('Strategy 1: Original Data (Red = Detected Anomaly)', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Transaction Amount')

# Capping effect
axes[0, 1].hist(df_original['transaction_amount'], bins=50, alpha=0.5, label='Original', color='blue')
axes[0, 1].hist(df_capped['transaction_amount'], bins=50, alpha=0.5, label='Capped (1%-99%)', color='orange')
axes[0, 1].set_title('Strategy 2: Capping (Winsorization)', fontsize=11, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].set_xlabel('Transaction Amount')

# Transformation effect
axes[1, 0].hist(df_original['transaction_amount'], bins=50, alpha=0.5, label='Original', color='blue')
axes[1, 0].hist(df_transformed['transaction_amount_log'], bins=50, alpha=0.5, label='Log Transformed', color='green')
axes[1, 0].set_title('Strategy 3: Log Transformation', fontsize=11, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].set_xlabel('Value')

# Flagging strategy
risk_summary = df_flagged.groupby('is_high_risk').agg({
    'transaction_amount': 'mean',
    'is_fraud': 'mean'
}).round(2)
risk_summary.plot(kind='bar', ax=axes[1, 1], color=['skyblue', 'coral'])
axes[1, 1].set_title('Strategy 4: Flagging for Separate Modeling', fontsize=11, fontweight='bold')
axes[1, 1].set_xlabel('Is High Risk (0=No, 1=Yes)')
axes[1, 1].set_ylabel('Average Value')
axes[1, 1].legend(['Avg Transaction Amount', 'Fraud Rate'])
axes[1, 1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print("📊 Strategy Comparison:")
print(f"   Original dataset size: {len(df_original)}")
print(f"   After removal: {len(df_removed)} (-{len(df_original) - len(df_removed)} samples)")
print(f"   Fraud rate in original: {df_original['is_fraud'].mean():.2%}")
print(f"   Fraud rate after removal: {df_removed['is_fraud'].mean():.2%}")
print("\n💡 Each strategy has trade-offs. Choose based on your use case!")

### 📋 Handling Strategy Decision Matrix

| Strategy | When to Use | Pros | Cons |
|----------|-------------|------|------|
| **Remove** | Anomalies are data errors | Clean dataset | May lose rare but important signals |
| **Cap** | Extreme values distort models | Retains all data | Artificially compresses distribution |
| **Transform** | Skewed distributions | Normalizes data | Changes interpretability |
| **Flag** | Anomalies are meaningful | Preserves information | Increases dimensionality |
| **Separate Model** | Different data generation process | Specialized handling | Complexity increases |

> **🔑 Key Recommendation**: In fraud detection, never blindly remove anomalies—they're often exactly what you're looking for!

<a id='section10'></a>
## 10. Best Practices and Decision Framework ✅

### Production ML Pipeline Checklist

In [ ]:
# =============================================================================
# Reusable Anomaly Detection Pipeline
# =============================================================================

class AnomalyDetectionPipeline:
    """
    Production-ready anomaly detection pipeline.
    """
    def __init__(self, contamination=0.05, random_state=42):
        self.contamination = contamination
        self.random_state = random_state
        self.scaler = StandardScaler()
        self.iso_forest = IsolationForest(
            n_estimators=100,
            contamination=contamination,
            random_state=random_state,
            n_jobs=-1
        )
        self.lof = LocalOutlierFactor(
            n_neighbors=20,
            contamination=contamination,
            novelty=True,
            n_jobs=-1
        )
        self.is_fitted = False
    
    def fit(self, X):
        """Fit the pipeline on training data."""
        X_scaled = self.scaler.fit_transform(X)
        self.iso_forest.fit(X_scaled)
        self.lof.fit(X_scaled)
        self.is_fitted = True
        return self
    
    def predict(self, X, method='ensemble'):
        """
        Predict anomalies.
        method: 'isolation_forest', 'lof', or 'ensemble'
        """
        if not self.is_fitted:
            raise ValueError("Pipeline must be fitted before prediction!")
        
        X_scaled = self.scaler.transform(X)
        
        if method == 'isolation_forest':
            return self.iso_forest.predict(X_scaled) == -1
        elif method == 'lof':
            return self.lof.predict(X_scaled) == -1
        elif method == 'ensemble':
            iso_pred = self.iso_forest.predict(X_scaled) == -1
            lof_pred = self.lof.predict(X_scaled) == -1
            return (iso_pred.astype(int) + lof_pred.astype(int)) >= 1
        else:
            raise ValueError(f"Unknown method: {method}")
    
    def get_anomaly_scores(self, X):
        """Get anomaly scores for ranking."""
        X_scaled = self.scaler.transform(X)
        iso_scores = -self.iso_forest.decision_function(X_scaled)
        return iso_scores

# Demonstrate the pipeline
pipeline = AnomalyDetectionPipeline(contamination=0.05)
pipeline.fit(X)

# Predict on the same data (in practice, use separate test set)
predictions = pipeline.predict(X, method='ensemble')
scores = pipeline.get_anomaly_scores(X)

print("✅ Production Pipeline Demonstration:")
print(f"   Anomalies detected: {predictions.sum()}")
print(f"   Top 5 anomaly scores: {np.sort(scores)[-5:][::-1]}")
print("\n🛡️ Pipeline ready for production deployment!")

# Best practices summary visualization
fig, ax = plt.subplots(figsize=(12, 8))
ax.axis('off')

best_practices = """
🛡️ ANOMALY DETECTION BEST PRACTICES 🛡️

1. DATA EXPLORATION PHASE
   ✅ Always visualize data before detection
   ✅ Understand domain context of anomalies
   ✅ Check for missing values and data quality issues

2. METHOD SELECTION
   ✅ Start with multiple methods (Isolation Forest + LOF)
   ✅ Use ensemble approaches for robustness
   ✅ Consider data size (LOF is O(n²), Isolation Forest is O(n log n))

3. PARAMETER TUNING
   ✅ Tune contamination based on domain knowledge
   ✅ Use cross-validation when labels available
   ✅ Monitor false positive rates in production

4. PRODUCTION DEPLOYMENT
   ✅ Retrain models periodically
   ✅ Monitor concept drift
   ✅ Log all anomaly decisions for auditing
   ✅ Implement feedback loops for model improvement

5. INTERPRETABILITY
   ✅ Always explain why something is anomalous
   ✅ Provide anomaly scores, not just binary labels
   ✅ Use dimensionality reduction for visualization

6. BUSINESS INTEGRATION
   ✅ Align anomaly threshold with business costs
   ✅ Consider human-in-the-loop for high-stakes decisions
   ✅ Document handling strategies clearly
"""

ax.text(0.05, 0.95, best_practices, transform=ax.transAxes, fontsize=11,
       verticalalignment='top', fontfamily='monospace',
       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.title('Complete Anomaly Detection Decision Framework', fontsize=14, fontweight='bold', pad=20)
plt.show()

<a id='exercises'></a>
## 🛠️ Hands-On Exercises

Now it's your turn! Complete these exercises to master anomaly detection.

### Exercise 1: Apply Isolation Forest
Apply Isolation Forest to the transaction dataset with different contamination values (0.01, 0.05, 0.10). Compare how the precision and recall change.

In [ ]:
# Exercise 1: Your code here



### Exercise 2: Visualize with t-SNE
Create a t-SNE visualization colored by the anomaly scores from Isolation Forest instead of binary labels. Use a continuous colormap to show the gradient of "anomalousness".

In [ ]:
# Exercise 2: Your code here



### Exercise 3: Compare All Methods
Create a comprehensive comparison table showing Precision, Recall, F1-Score, and False Positive Rate for all methods covered in this notebook (Z-Score, Modified Z, Isolation Forest, LOF, DBSCAN, Ensemble).

In [ ]:
# Exercise 3: Your code here



### Exercise 4: Handling Strategy Analysis
Train a simple classifier (e.g., Logistic Regression) to predict fraud using: (a) original data, (b) data with outliers removed, (c) data with outliers capped. Compare the AUC scores.

In [ ]:
# Exercise 4: Your code here



### Exercise 5: Build a Reusable Function
Create a function `detect_anomalies(data, method='ensemble', **kwargs)` that can accept any dataset and detection method, and returns a DataFrame with anomaly flags and scores.

In [ ]:
# Exercise 5: Your code here



### Exercise 6: Multivariate Analysis
Identify transactions that are NOT outliers in any single feature but ARE outliers when considering the combination of `transaction_amount` and `transactions_per_day` together (use Mahalanobis distance).

In [ ]:
# Exercise 6: Your code here



### Exercise 7: Time-Based Anomalies
Analyze the `transaction_time` feature to detect anomalies in temporal patterns. Are there transactions at unusual hours that might indicate fraud?

In [ ]:
# Exercise 7: Your code here



### Exercise 8: Feature Importance for Anomalies
Use PCA to determine which features contribute most to the anomalous behavior. Create a bar chart showing feature importance based on PCA loadings.

In [ ]:
# Exercise 8: Your code here



### Exercise 9: Create Anomaly Report
Generate a comprehensive PDF-style report (as a structured DataFrame) showing for each detected anomaly: its index, anomaly score, which method(s) detected it, and recommended action.

In [ ]:
# Exercise 9: Your code here



### Exercise 10: Real-World Scenario
You have a new dataset with 10,000 transactions and no labels. Design and implement a complete anomaly detection pipeline that:
1. Automatically selects the best method based on data characteristics
2. Generates visualizations
3. Outputs a risk-ranked list of transactions for human review
4. Includes confidence intervals

In [ ]:
# Exercise 10: Your code here



<a id='solutions'></a>
## Solutions & Key Insights (Review After Attempting)

### Solution 1: Isolation Forest with Different Contamination
```python
contamination_values = [0.01, 0.05, 0.10]
results = []

for cont in contamination_values:
    iso = IsolationForest(contamination=cont, random_state=42)
    preds = iso.fit_predict(X_scaled) == -1
    precision = df[preds]['is_fraud'].mean()
    recall = df.loc[df['is_fraud'] == 1, preds].mean()
    results.append({'contamination': cont, 'precision': precision, 'recall': recall})
```
**Key Insight**: Lower contamination increases precision (fewer false positives) but decreases recall. The optimal value depends on your business cost of false positives vs false negatives.

### Solution 2: Continuous t-SNE Visualization
```python
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=df['iso_forest_score'], 
           cmap='Reds', alpha=0.6)
plt.colorbar(label='Anomaly Score')
```
**Key Insight**: Continuous scores allow you to set dynamic thresholds rather than binary decisions. This is crucial for risk-based decision making.

### Solution 3: Comprehensive Comparison
The ensemble method typically provides the best balance. However, if computational efficiency is critical, Isolation Forest alone provides 80% of the benefit with 20% of the computational cost.

### Solution 4: Impact on Model Performance
In fraud detection, removing outliers often HURTS performance because outliers ARE the fraud cases. This contrasts with traditional ML where outliers are often noise.

### Solution 5: Reusable Function Pattern
Always include input validation, proper error handling, and clear documentation. Consider making the function work with both pandas DataFrames and numpy arrays.

### Solution 6: Multivariate vs Univariate
Approximately 15-20% of fraud cases are only detectable through multivariate analysis. These are sophisticated frauds designed to evade simple rule-based systems.

### Solution 7: Temporal Patterns
Fraud often occurs at unusual hours (2-5 AM) when legitimate customers are sleeping. Time-based features are powerful but often overlooked.

### Solution 8: PCA Feature Importance
The first principal component typically explains 40-60% of variance in transaction data. Features with highest loadings on this component are most indicative of anomalous behavior.

### Solution 9: Anomaly Report Structure
Include: Detection timestamp, anomaly score percentile, contributing features (which features were most anomalous), and recommended action based on business rules.

### Solution 10: Production Pipeline
A robust pipeline should include: data validation, automatic method selection based on data size and dimensionality, ensemble voting, confidence scoring, and human review queuing.